# MLP vs LSTM vs TikTok: honest portfolio comparison

This notebook validates the Week 2 models against the Week 1 TikTok strategy on the same EGX universe, calendar, simulator, benchmark, and commission. Hyperparameters are selected on a validation period; the final test period is opened only after the choice is made.

The final chart uses growth of 1,000 EGP, and the seed table shows whether a good curve survives a changed random seed.

In [ ]:
import os
import sys
from pathlib import Path

while not os.path.isdir('src') and os.path.dirname(os.getcwd()) != os.getcwd():
    os.chdir('..')
sys.path.insert(0, 'src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from tradinglab.data_feed import DataFeed, load_egx30_returns
from tradinglab.features import build_pooled_dataset, build_pooled_sequences
from tradinglab.models import MLP, DeepMLP, LSTMRegressor
from tradinglab.ml import train_model, predict
from tradinglab.backtester import run_backtest
from tradinglab.simulator import PortfolioSimulator
from tradinglab.strategies.sma import sma_crossover_weights

In [ ]:
DATA_DIR = Path('data/egx')
EGX30_PATH = Path('data/egx30.csv')
LOOKBACK = 30
COMMISSION = 0.005
TOP_K = 4
EPOCHS = 100
SEEDS = [0, 7, 42]
TRAIN_END_DATE = pd.Timestamp('2022-01-01')
VALIDATION_END_DATE = pd.Timestamp('2022-07-01')

feed = DataFeed.from_dir(DATA_DIR)
train_end = int(feed.dates.searchsorted(TRAIN_END_DATE))
validation_end = int(feed.dates.searchsorted(VALIDATION_END_DATE))
test_start = validation_end

sim = PortfolioSimulator(feed, benchmark='equal_weight', commission=COMMISSION)
print(f'{feed.n_assets} stocks | {feed.n_days} common trading days')
print(f'train: {feed.dates[0].date()} -> {feed.dates[train_end - 1].date()}')
print(f'validation: {feed.dates[train_end].date()} -> {feed.dates[validation_end - 1].date()}')
print(f'test/plots: {feed.dates[test_start].date()} -> {feed.dates[-1].date()}')

This comparison maps the requested source files to the corresponding implementations:

- TikTok: `week1/06-tiktok-strategy/tiktok_strategy.py` and `week1/05-dashboard/s_tik.ipynb`
- MLP: `week2/03-form-prediction-to-portfolio/mlp.ipynb` and `week2/01-mlp/day_1.ipynb`
- LSTM: `week2/03-form-prediction-to-portfolio/lstm.ipynb` and `week2/02-lstm/day_2.ipynb`
- SMA: the Week 1 `sma_crossover_weights` baseline
- Benchmark: the shared equal-weight benchmark, plus the real EGX30 curve when available

All strategies are evaluated through the same `DataFeed`, `PortfolioSimulator`, `run_backtest`, dates, test period, and commission so the curves are comparable.

In [ ]:
def predictions_to_weights(predicted_returns, top_k=TOP_K):
    weights = np.zeros(len(predicted_returns), dtype=np.float32)
    positive = np.flatnonzero(predicted_returns > 0)
    if len(positive) == 0:
        return weights
    selected = positive[np.argsort(predicted_returns[positive])[-min(top_k, len(positive)):]]
    weights[selected] = 1.0 / len(selected)
    return weights

def mlp_strategy(model, top_k=TOP_K):
    def strategy(observation):
        predictions = predict(model, observation[:, -1, :].astype('float32'))
        return predictions_to_weights(predictions, top_k)
    return strategy

def lstm_strategy(model, seq_len, top_k=TOP_K):
    def strategy(observation):
        window = observation[:, -seq_len:, :].astype('float32')
        predictions = predict(model, window)
        return predictions_to_weights(predictions, top_k)
    return strategy

def make_tiktok_strategy(week_days=5, sensitivity=1.0):
    state = {'weights': None, 'day_count': 0}
    def strategy(observation):
        n_assets = observation.shape[0]
        current = state['weights']
        if current is None or current.sum() == 0:
            current = np.ones(n_assets, dtype=np.float32) / n_assets
            state['weights'] = current
        if state['day_count'] % week_days == 0:
            recent_returns = observation[:, -week_days:, 0]
            return_nd = np.prod(1 + recent_returns, axis=1) - 1
            new_weights = current * (1 - return_nd * sensitivity)
            new_weights = np.clip(new_weights, 0, None)
            total = new_weights.sum()
            current = np.zeros(n_assets) if total <= 0 else new_weights / total
            state['weights'] = current
        state['day_count'] += 1
        return state['weights']
    return strategy

## Validation search

We try different widths, depths, sequence lengths, learning rates, and seeds. The validation equity is used to select one configuration per model family. The test period is not used in this choice.

In [ ]:
def seed_everything(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)

def fit_mlp(config, seed, split_day=train_end):
    X_train, y_train, X_eval, y_eval = build_pooled_dataset(feed, split_day)
    seed_everything(seed)
    if config['depth'] == 1:
        model = MLP(X_train.shape[1], hidden=config['hidden'])
    else:
        model = DeepMLP(X_train.shape[1], hidden=config['hidden'], n_hidden_layers=config['depth'])
    history = train_model(model, X_train, y_train, X_eval, y_eval, epochs=EPOCHS, lr=config['lr'])
    return model, history

def fit_lstm(config, seed, split_day=train_end):
    X_train, y_train, X_eval, y_eval = build_pooled_sequences(feed, split_day, seq_len=config['seq_len'])
    seed_everything(seed)
    model = LSTMRegressor(n_features=X_train.shape[2], hidden=config['hidden'])
    history = train_model(model, X_train, y_train, X_eval, y_eval, epochs=EPOCHS, lr=config['lr'])
    return model, history

mlp_grid = [
    {'hidden': 16, 'depth': 1, 'lr': 1e-3},
    {'hidden': 32, 'depth': 1, 'lr': 1e-3},
    {'hidden': 64, 'depth': 1, 'lr': 5e-4},
    {'hidden': 32, 'depth': 2, 'lr': 1e-3},
]
lstm_grid = [
    {'hidden': 16, 'seq_len': 5, 'lr': 1e-3},
    {'hidden': 32, 'seq_len': 5, 'lr': 1e-3},
    {'hidden': 32, 'seq_len': 10, 'lr': 1e-3},
    {'hidden': 64, 'seq_len': 10, 'lr': 5e-4},
]

validation_rows = []
for family, grid, fitter in [('MLP', mlp_grid, fit_mlp), ('LSTM', lstm_grid, fit_lstm)]:
    for config_id, config in enumerate(grid):
        scores = []
        for seed in SEEDS:
            model, history = fitter(config, seed)
            strategy = mlp_strategy(model) if family == 'MLP' else lstm_strategy(model, config['seq_len'])
            result = run_backtest(sim, strategy, LOOKBACK, start=train_end, end=validation_end)
            scores.append(result['portfolio'][-1])
        validation_rows.append({'family': family, 'config_id': config_id, 'config': config, 'mean_validation': np.mean(scores), 'std_validation': np.std(scores), 'seed_scores': scores})

validation_table = pd.DataFrame(validation_rows).sort_values(['family', 'mean_validation'], ascending=[True, False])
display(validation_table[['family', 'config_id', 'config', 'mean_validation', 'std_validation', 'seed_scores']])

In [ ]:
best_mlp_id = validation_table.loc[validation_table['family'].eq('MLP'), 'mean_validation'].idxmax()
best_lstm_id = validation_table.loc[validation_table['family'].eq('LSTM'), 'mean_validation'].idxmax()
best_mlp = validation_table.loc[best_mlp_id, 'config']
best_lstm = validation_table.loc[best_lstm_id, 'config']
print('Selected MLP:', best_mlp)
print('Selected LSTM:', best_lstm)

## Final test backtest

Each selected configuration is retrained from scratch on the full train period, then evaluated on the untouched test period for every seed. TikTok and SMA use the same test dates and simulator.

In [ ]:
def curve_metrics(result):
    curve = np.asarray(result['portfolio'])
    benchmark = np.asarray(result['benchmark'])
    daily = np.asarray(result['portfolio_returns'])
    drawdown = curve / np.maximum.accumulate(curve) - 1
    sharpe = np.sqrt(252) * daily.mean() / daily.std() if daily.std() else np.nan
    return {
        'final_egp': curve[-1] * 1000,
        'return_pct': (curve[-1] - 1) * 100,
        'benchmark_return_pct': (benchmark[-1] - 1) * 100,
        'alpha_pct': (curve[-1] - benchmark[-1]) * 100,
        'sharpe': sharpe,
        'max_drawdown_pct': drawdown.min() * 100,
        'beat_benchmark': curve[-1] > benchmark[-1],
    }

final_results = {}
for seed in SEEDS:
    mlp_model, _ = fit_mlp(best_mlp, seed, split_day=validation_end)
    lstm_model, _ = fit_lstm(best_lstm, seed, split_day=validation_end)
    final_results[('MLP', seed)] = run_backtest(sim, mlp_strategy(mlp_model), LOOKBACK, start=test_start)
    final_results[('LSTM', seed)] = run_backtest(sim, lstm_strategy(lstm_model, best_lstm['seq_len']), LOOKBACK, start=test_start)

final_results[('TikTok', 'fixed')] = run_backtest(sim, make_tiktok_strategy(week_days=5), LOOKBACK, start=test_start)
final_results[('SMA week1', 'fixed')] = run_backtest(sim, lambda observation: sma_crossover_weights(observation, 9, 20), LOOKBACK, start=test_start)

metric_rows = []
for (strategy_name, seed), result in final_results.items():
    row = {'strategy': strategy_name, 'seed': seed}
    row.update(curve_metrics(result))
    metric_rows.append(row)
metrics = pd.DataFrame(metric_rows)
display(metrics)

In [ ]:
START = 1000.0
plt.figure(figsize=(13, 6))

for seed in SEEDS:
    result = final_results[('MLP', seed)]
    plt.plot(result['dates'], result['portfolio'] * START, alpha=0.25, color='tab:blue')
    result = final_results[('LSTM', seed)]
    plt.plot(result['dates'], result['portfolio'] * START, alpha=0.25, color='tab:orange')

mlp_mean = np.mean([final_results[('MLP', seed)]['portfolio'] for seed in SEEDS], axis=0)
lstm_mean = np.mean([final_results[('LSTM', seed)]['portfolio'] for seed in SEEDS], axis=0)
dates = final_results[('MLP', SEEDS[0])]['dates']
plt.plot(dates, mlp_mean * START, color='tab:blue', linewidth=2.2, label='MLP (day_1) mean')
plt.plot(dates, lstm_mean * START, color='tab:orange', linewidth=2.2, label='LSTM (day_2) mean')
plt.plot(dates, final_results[('TikTok', 'fixed')]['portfolio'] * START, color='tab:green', linewidth=1.8, label='TikTok (s_tik)')
plt.plot(dates, final_results[('SMA week1', 'fixed')]['portfolio'] * START, color='tab:red', linewidth=1.5, label='SMA crossover (week 1)')
plt.plot(dates, final_results[('MLP', SEEDS[0])]['benchmark'] * START, color='black', linestyle='--', linewidth=1.6, label='equal-weight benchmark')

egx30_returns = load_egx30_returns(EGX30_PATH, feed.dates)
if egx30_returns is not None:
    egx30_test = egx30_returns[test_start + 1:]
    egx30_curve = np.cumprod(1 + egx30_test) * START
    plt.plot(dates, egx30_curve, color='purple', linestyle=':', linewidth=1.8, label='real EGX30')

plt.title('MLP vs LSTM vs TikTok vs benchmark')
plt.ylabel('Portfolio value (EGP)')
plt.xlabel('Date')
plt.legend()
plt.grid(alpha=0.3)
plt.gcf().autofmt_xdate()
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor


def legacy_lag_data():
    rows = []
    for asset, symbol in enumerate(feed.symbols):
        returns = feed.returns[:, asset]
        for day in range(5, feed.n_days - 1):
            rows.append((day, asset, [returns[day - lag] for lag in range(1, 6)], returns[day]))
    return rows


legacy_rows = legacy_lag_data()
legacy_train = [row for row in legacy_rows if row[0] < validation_end]
legacy_test = [row for row in legacy_rows if row[0] >= validation_end]
legacy_scaler = StandardScaler().fit(np.asarray([row[2] for row in legacy_train]))
legacy_X_train = legacy_scaler.transform(np.asarray([row[2] for row in legacy_train])).astype('float32')
legacy_y_train = np.asarray([row[3] for row in legacy_train], dtype='float32')


def legacy_mlp_strategy(model):
    def strategy(observation):
        features = np.column_stack([
            observation[:, -offset, 0] for offset in range(1, 6)
        ])
        predictions = model.predict(legacy_scaler.transform(features))
        return predictions_to_weights(predictions)
    return strategy


def legacy_lstm_strategy(model):
    def strategy(observation):
        features = np.column_stack([
            observation[:, -offset, 0] for offset in range(1, 6)
        ]).astype('float32')
        predictions = predict(model, legacy_scaler.transform(features).reshape(len(features), 5, 1))
        return predictions_to_weights(predictions)
    return strategy


def fit_legacy_mlp(seed):
    model = MLPRegressor(
        hidden_layer_sizes=(32,),
        activation='relu',
        solver='adam',
        random_state=seed,
        shuffle=False,
        max_iter=1,
        warm_start=True,
    )
    for _ in range(EPOCHS):
        model.fit(legacy_X_train, legacy_y_train)
    return model


class LegacyLSTM(torch.nn.Module):
    def __init__(self, hidden=64):
        super().__init__()
        self.lstm = torch.nn.LSTM(input_size=1, hidden_size=hidden, batch_first=True)
        self.fc = torch.nn.Linear(hidden, 1)

    def forward(self, x):
        output, _ = self.lstm(x)
        return self.fc(output[:, -1, :]).squeeze(-1)


def fit_legacy_lstm(seed):
    seed_everything(seed)
    X = torch.tensor(legacy_X_train.reshape(len(legacy_X_train), 5, 1))
    y = torch.tensor(legacy_y_train)
    model = LegacyLSTM(hidden=64)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = torch.nn.MSELoss()
    for _ in range(EPOCHS):
        optimizer.zero_grad()
        loss = loss_fn(model(X), y)
        loss.backward()
        optimizer.step()
    return model


legacy_results = {}
for seed in SEEDS:
    legacy_mlp = fit_legacy_mlp(seed)
    legacy_lstm = fit_legacy_lstm(seed)
    legacy_results[('MLP day_1', seed)] = run_backtest(
        sim, legacy_mlp_strategy(legacy_mlp), LOOKBACK, start=test_start
    )
    legacy_results[('LSTM day_2', seed)] = run_backtest(
        sim, legacy_lstm_strategy(legacy_lstm), LOOKBACK, start=test_start
    )

legacy_metric_rows = []
for (strategy_name, seed), legacy_result in legacy_results.items():
    row = {'strategy': strategy_name, 'seed': seed}
    row.update(curve_metrics(legacy_result))
    legacy_metric_rows.append(row)
metrics = pd.concat([metrics, pd.DataFrame(legacy_metric_rows)], ignore_index=True)

print('Legacy MLP/LSTM curves added:', len(legacy_results))

In [ ]:
s_tik_full = run_s_tik_native(feed.dates[test_start])
s_tik_mask = s_tik_full['dates'] >= feed.dates[test_start]
s_tik_result = {
    'dates': s_tik_full['dates'][s_tik_mask],
    'portfolio': s_tik_full['portfolio'][s_tik_mask],
    'benchmark': s_tik_full['benchmark'][s_tik_mask],
}
s_tik_result['portfolio'] /= s_tik_result['portfolio'][0]
s_tik_result['benchmark'] /= s_tik_result['benchmark'][0]


def align_curve(result, target_dates, key='portfolio'):
    series = pd.Series(result[key], index=pd.DatetimeIndex(result['dates']))
    return series.reindex(target_dates, method='ffill').bfill().to_numpy()


def weekly_view(result, target_dates):
    return align_curve(result, target_dates)


dates = final_results[('MLP', SEEDS[0])]['dates']
tiktok_weekly = weekly_view(final_results[('TikTok', 'fixed')], s_tik_result['dates'])

egx30_returns = load_egx30_returns(EGX30_PATH, feed.dates)
egx30_curve = np.cumprod(1 + egx30_returns[test_start + 1:])
egx30_curve = egx30_curve / egx30_curve[0]
egx30_weekly = pd.Series(egx30_curve, index=dates).reindex(
    s_tik_result['dates'], method='ffill'
).bfill().to_numpy()

plt.figure(figsize=(13, 6))
plt.plot(s_tik_result['dates'], s_tik_result['portfolio'] * START, label='s_tik.ipynb weekly buy/sell', linewidth=2)
plt.plot(s_tik_result['dates'], tiktok_weekly * START, label='tiktok_strategy.py weekly tilt', linewidth=2)
plt.plot(s_tik_result['dates'], s_tik_result['benchmark'] * START, label='s_tik benchmark', linestyle='--')
plt.plot(s_tik_result['dates'], egx30_weekly * START, label='real EGX30', linestyle=':', linewidth=1.8)
plt.title('TikTok comparison: s_tik.ipynb vs tiktok_strategy.py vs real EGX30')
plt.ylabel('Portfolio value (EGP)')
plt.xlabel('Date')
plt.legend(); plt.grid(alpha=0.3)
plt.gcf().autofmt_xdate(); plt.tight_layout(); plt.show()

mlp_legacy_mean = np.mean([align_curve(legacy_results[('MLP day_1', seed)], dates) for seed in SEEDS], axis=0)
mlp_current_mean = np.mean([align_curve(final_results[('MLP', seed)], dates) for seed in SEEDS], axis=0)
lstm_legacy_mean = np.mean([align_curve(legacy_results[('LSTM day_2', seed)], dates) for seed in SEEDS], axis=0)
lstm_current_mean = np.mean([align_curve(final_results[('LSTM', seed)], dates) for seed in SEEDS], axis=0)
mlp_benchmark = align_curve(final_results[('MLP', SEEDS[0])], dates, key='benchmark')
lstm_benchmark = align_curve(final_results[('LSTM', SEEDS[0])], dates, key='benchmark')

plt.figure(figsize=(13, 6))
plt.plot(dates, mlp_legacy_mean * START, label='MLP day_1.ipynb', linewidth=2)
plt.plot(dates, mlp_current_mean * START, label='MLP mlp.ipynb / lstm.ipynb', linewidth=2)
plt.plot(dates, mlp_benchmark * START, label='equal-weight benchmark', linestyle='--')
plt.plot(dates, egx30_curve * START, label='real EGX30', linestyle=':', linewidth=1.8)
plt.title('MLP comparison: day_1.ipynb vs mlp.ipynb vs real EGX30')
plt.ylabel('Portfolio value (EGP)'); plt.xlabel('Date')
plt.legend(); plt.grid(alpha=0.3)
plt.gcf().autofmt_xdate(); plt.tight_layout(); plt.show()

plt.figure(figsize=(13, 6))
plt.plot(dates, lstm_legacy_mean * START, label='LSTM day_2.ipynb', linewidth=2)
plt.plot(dates, lstm_current_mean * START, label='LSTM lstm.ipynb', linewidth=2)
plt.plot(dates, lstm_benchmark * START, label='equal-weight benchmark', linestyle='--')
plt.plot(dates, egx30_curve * START, label='real EGX30', linestyle=':', linewidth=1.8)
plt.title('LSTM comparison: day_2.ipynb vs lstm.ipynb vs real EGX30')
plt.ylabel('Portfolio value (EGP)'); plt.xlabel('Date')
plt.legend(); plt.grid(alpha=0.3)
plt.gcf().autofmt_xdate(); plt.tight_layout(); plt.show()

benchmark_tiktok = align_curve(final_results[('TikTok', 'fixed')], dates, key='benchmark')
benchmark_sma = align_curve(final_results[('SMA week1', 'fixed')], dates, key='benchmark')

plt.figure(figsize=(13, 6))
plt.plot(dates, benchmark_tiktok * START, label='PortfolioSimulator equal-weight')
plt.plot(dates, benchmark_sma * START, label='SMA backtest benchmark', linestyle='--')
plt.plot(s_tik_result['dates'], s_tik_result['benchmark'] * START, label='s_tik benchmark', linestyle=':')
plt.plot(dates, egx30_curve * START, label='real EGX30', linewidth=2)
plt.title('Benchmark comparison: equal-weight vs real EGX30')
plt.ylabel('Portfolio value (EGP)'); plt.xlabel('Date')
plt.legend(); plt.grid(alpha=0.3)
plt.gcf().autofmt_xdate(); plt.tight_layout(); plt.show()

source_comparison = pd.DataFrame([
    {'family': 'TikTok', 'source_a': 's_tik.ipynb', 'source_b': 'tiktok_strategy.py', 'final_a': s_tik_result['portfolio'][-1] * START, 'final_b': tiktok_weekly[-1] * START},
    {'family': 'MLP', 'source_a': 'day_1.ipynb', 'source_b': 'mlp.ipynb', 'final_a': mlp_legacy_mean[-1] * START, 'final_b': mlp_current_mean[-1] * START},
    {'family': 'LSTM', 'source_a': 'day_2.ipynb', 'source_b': 'lstm.ipynb', 'final_a': lstm_legacy_mean[-1] * START, 'final_b': lstm_current_mean[-1] * START},
])
display(source_comparison)

In [ ]:
summary = metrics.groupby('strategy', sort=False).agg(
    mean_final_egp=('final_egp', 'mean'),
    std_final_egp=('final_egp', 'std'),
    mean_alpha_pct=('alpha_pct', 'mean'),
    worst_alpha_pct=('alpha_pct', 'min'),
    beat_count=('beat_benchmark', 'sum'),
    runs=('beat_benchmark', 'count'),
).sort_values('mean_final_egp', ascending=False)

# The benchmark is the same in every test result, so these answer the direct questions.
mean_final = summary['mean_final_egp']
head_to_head = pd.DataFrame({
    'mean_final_egp': mean_final,
    'beats_MLP': mean_final > mean_final['MLP'],
    'beats_LSTM': mean_final > mean_final['LSTM'],
    'beats_SMA': mean_final > mean_final['SMA week1'],
    'beats_TikTok': mean_final > mean_final['TikTok'],
    'beats_equal_weight': summary['mean_alpha_pct'] > 0,
})

display(summary)
display(head_to_head)

best_strategy = summary.index[0]
print('Best mean final value:', best_strategy)
print('LSTM beats MLP:', mean_final['LSTM'] > mean_final['MLP'])
print('TikTok beats SMA:', mean_final['TikTok'] > mean_final['SMA week1'])
print('The seed test matters: a strategy is robust only if its alpha stays positive across seeds, not just on one curve.')